In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ngocccchannn/evaluator/evaluate.py
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/test_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/train_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/val_cold.parquet


In [2]:
import os
import sys 
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import json
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")


sys.path.append('/kaggle/input/datasets/ngocccchannn/evaluator') 

from evaluate import evaluate

Sử dụng thiết bị: cuda


In [3]:
print("Đang tải dữ liệu...")
train_df = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/cold-scenario-data/train_cold.parquet')
#val_df = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/cold-scenario-data/val_cold.parquet')

#with open('/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_val.json', 'r') as f:
#    cold_items_val = json.load(f)

#with open('/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json', 'r') as f:
#    cold_items_test = json.load(f)

# Ánh xạ ID người dùng (user_id) và khóa học (course_id) sang chỉ số nguyên liên tục [0, N-1]
#all_users = pd.concat([train_df['user_id'], val_df['user_id']]).unique()
#all_items = pd.concat([train_df['course_id'], val_df['course_id']]).unique()
all_users = train_df['user_id'].unique()
all_items = train_df['course_id'].unique()

user2id = {str(u): i for i, u in enumerate(all_users)}
item2id = {str(item): i for i, item in enumerate(all_items)}

num_users = len(user2id)
num_items = len(item2id)
print(f"Tổng số User: {num_users}, Tổng số Item: {num_items}")

Đang tải dữ liệu...
Tổng số User: 98220, Tổng số Item: 2401


In [4]:
# Chuẩn bị dữ liệu tương tác cho train
train_users = torch.tensor([user2id[str(u)] for u in train_df['user_id']], dtype=torch.long)
train_items = torch.tensor([item2id[str(i)] for i in train_df['course_id']], dtype=torch.long)

# Tách 5% tương tác train để làm tập validation nội bộ (theo yêu cầu Review)
val_split_idx = int(len(train_users) * 0.95)
internal_train_u = train_users[:val_split_idx]
internal_train_i = train_items[:val_split_idx]

val_int_u = train_users[val_split_idx:]
val_int_i = train_items[val_split_idx:]

# Tạo cấu trúc edge_index cho đồ thị PyTorch Geometric / Sparse Tensor
# LightGCN cần đồ thị hai phía (user-item)
edge_index_u = torch.cat([internal_train_u, internal_train_i + num_users])
edge_index_i = torch.cat([internal_train_i + num_users, internal_train_u])
edge_index = torch.stack([edge_index_u, edge_index_i], dim=0).to(DEVICE)

# Xây dựng ground-truth target và mask cho tập validation nội bộ (để đánh giá Recall@20)
val_target_dict = {}
for u, i in zip(val_int_u.numpy(), val_int_i.numpy()):
    if u not in val_target_dict:
        val_target_dict[int(u)] = []
    val_target_dict[int(u)].append(int(i))

In [5]:
class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, latent_dim=64, n_layers=3):
        super(LightGCN, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.latent_dim = latent_dim
        self.n_layers = n_layers
        
        self.user_embedding = nn.Embedding(num_users, latent_dim)
        self.item_embedding = nn.Embedding(num_items, latent_dim)
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)

    def forward(self, adj_matrix): 
        users_emb = self.user_embedding.weight
        items_emb = self.item_embedding.weight
        all_emb = torch.cat([users_emb, items_emb], dim=0)
        
        embs = [all_emb]
        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(adj_matrix, all_emb)
            embs.append(all_emb)
            
        lightgcn_emb = torch.stack(embs, dim=1).mean(dim=1)
        users_emb_final, items_emb_final = torch.split(lightgcn_emb, [self.num_users, self.num_items])
        return users_emb_final, items_emb_final

In [6]:
# Xây dựng ma trận kề chuẩn hóa cho LightGCN
print("Đang xây dựng ma trận kề cho đồ thị...")
row = torch.cat([internal_train_u, internal_train_i + num_users])
col = torch.cat([internal_train_i + num_users, internal_train_u])
val = torch.ones(row.size(0), dtype=torch.float32)

# Tính bậc (degree) để chuẩn hóa đối xứng D^{-1/2} A D^{-1/2}
assoc = torch.sparse_coo_tensor(torch.stack([row, col]), val, (num_users + num_items, num_users + num_items)).to(DEVICE)
deg = torch.sparse.sum(assoc, dim=1).to_dense()

deg_inv_sqrt = torch.pow(deg, -0.5)
deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0.0

# Tạo chỉ số trên cùng thiết bị và mở rộng thành dạng 2 chiều [2, N] cho ma trận đường chéo thưa
i_idx = torch.arange(num_users + num_items, device=DEVICE)
diag_indices = torch.stack([i_idx, i_idx], dim=0)

deg_mat_inv_sqrt = torch.sparse_coo_tensor(
    diag_indices, 
    deg_inv_sqrt.to(DEVICE), 
    (num_users + num_items, num_users + num_items)
)

adj_mat = torch.sparse.mm(deg_mat_inv_sqrt, torch.sparse.mm(assoc, deg_mat_inv_sqrt))

Đang xây dựng ma trận kề cho đồ thị...


/tmp/ipykernel_23/2075432310.py:8: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  assoc = torch.sparse_coo_tensor(torch.stack([row, col]), val, (num_users + num_items, num_users + num_items)).to(DEVICE)


In [7]:
LATENT_DIM = 64  # d = 64
N_LAYERS = 3     # L = 3
BATCH_SIZE = 1024
LR = 1e-3

model = LightGCN(num_users, num_items, latent_dim=LATENT_DIM, n_layers=N_LAYERS).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)

def bpr_loss(users, pos_items, neg_items, u_emb, i_emb):
    u_g = u_emb[users]
    pos_g = i_emb[pos_items]
    neg_g = i_emb[neg_items]
    pos_scores = torch.sum(u_g * pos_g, dim=1)
    neg_scores = torch.sum(u_g * neg_g, dim=1)
    loss = -torch.mean(torch.nn.functional.logsigmoid(pos_scores - neg_scores))
    return loss

print("--- Bắt đầu huấn luyện LightGCN nền chung ---")
best_recall = 0.0
best_weights = None
epochs = 40

for epoch in range(epochs):
    model.train()
    total_loss = 0
    # Tạo negative sampling ngẫu nhiên cho BPR loss
    perm = torch.randperm(len(internal_train_u))
    sampled_u = internal_train_u[perm]
    sampled_pos_i = internal_train_i[perm]
    sampled_neg_i = torch.randint(0, num_items, (len(perm),), dtype=torch.long)
    
    for s in range(0, len(perm), BATCH_SIZE):
        batch_u = sampled_u[s:s+BATCH_SIZE].to(DEVICE)
        batch_pos = sampled_pos_i[s:s+BATCH_SIZE].to(DEVICE)
        batch_neg = sampled_neg_i[s:s+BATCH_SIZE].to(DEVICE)
        
        optimizer.zero_grad()
        u_emb, i_emb = model(adj_mat)
        loss = bpr_loss(batch_u, batch_pos, batch_neg, u_emb, i_emb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    # Đánh giá Early Stopping theo Recall@20 trên tập validation nội bộ bằng evaluator chung
    model.eval()
    with torch.no_grad():
        u_emb, i_emb = model(adj_mat)
        def score_fn(users_tensor):
            u_idx = users_tensor.to(DEVICE)
            return torch.matmul(u_emb[u_idx], i_emb.t())
        
        # Gọi hàm evaluate chuẩn từ evaluator.py
        val_res = evaluate(score_fn, val_target_dict, mask={}, ks=(20,), bs=1024)
        current_recall = val_res['Recall@20'].mean()
        
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss:.4f} | Recall@20 (Val): {current_recall:.4f}")
    
    if current_recall > best_recall:
        best_recall = current_recall
        best_weights = model.state_dict().copy()

# Lưu checkpoint tốt nhất
torch.save(best_weights, 'lightgcn_base_best.pt')
print(f"Huấn luyện hoàn thành! Best Recall@20: {best_recall:.4f}")

--- Bắt đầu huấn luyện LightGCN nền chung ---
Epoch 1/40 | Loss: 135.6652 | Recall@20 (Val): 0.0978
Epoch 2/40 | Loss: 87.3616 | Recall@20 (Val): 0.1101
Epoch 3/40 | Loss: 74.7369 | Recall@20 (Val): 0.1203
Epoch 4/40 | Loss: 66.4482 | Recall@20 (Val): 0.1236
Epoch 5/40 | Loss: 60.6233 | Recall@20 (Val): 0.1233
Epoch 6/40 | Loss: 56.3684 | Recall@20 (Val): 0.1249
Epoch 7/40 | Loss: 52.9738 | Recall@20 (Val): 0.1251
Epoch 8/40 | Loss: 50.1054 | Recall@20 (Val): 0.1244
Epoch 9/40 | Loss: 47.3278 | Recall@20 (Val): 0.1279
Epoch 10/40 | Loss: 45.5229 | Recall@20 (Val): 0.1284
Epoch 11/40 | Loss: 43.3650 | Recall@20 (Val): 0.1313
Epoch 12/40 | Loss: 40.9340 | Recall@20 (Val): 0.1340
Epoch 13/40 | Loss: 40.0640 | Recall@20 (Val): 0.1342
Epoch 14/40 | Loss: 38.2077 | Recall@20 (Val): 0.1345
Epoch 15/40 | Loss: 37.0168 | Recall@20 (Val): 0.1344
Epoch 16/40 | Loss: 35.7174 | Recall@20 (Val): 0.1388
Epoch 17/40 | Loss: 34.5126 | Recall@20 (Val): 0.1379
Epoch 18/40 | Loss: 33.4120 | Recall@20 (Val

In [8]:
model.load_state_dict(best_weights)
model.eval()
with torch.no_grad():
    u_emb, i_emb = model(adj_mat)
    np.save('g_u.npy', u_emb.cpu().numpy())
    np.save('g_i.npy', i_emb.cpu().numpy())

# Lấy Git Commit Hash để đảm bảo tính tái lập (theo yêu cầu Mục 6.1)
try:
    commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD']).strip().decode('utf-8')
except Exception:
    commit_hash = "unknown_commit"

print(f"Đã lưu thành công file embedding: 'g_u.npy' và 'g_i.npy'.")
print(f"Git Commit Hash hiện tại: {commit_hash}")

Đã lưu thành công file embedding: 'g_u.npy' và 'g_i.npy'.
Git Commit Hash hiện tại: unknown_commit


fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
